<a href="https://colab.research.google.com/github/samyakbaid/ML_coding_questions/blob/main/A1_Q1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from sklearn.datasets import load_diabetes

SEED = 42
np.random.seed(SEED)

In [ ]:
raw   = load_diabetes()
X_all = raw.data
y_all = raw.target

FEATURE_NAMES = raw.feature_names
N, D = X_all.shape

print(f"Dataset shape  : {X_all.shape}")
print(f"Feature names  : {FEATURE_NAMES}")
print(f"Target — min={y_all.min():.0f}  max={y_all.max():.0f}  mean={y_all.mean():.1f}")

Dataset shape  : (442, 10)
Feature names  : ['age', 'sex', 'bmi', 'bp', 's1', 's2', 's3', 's4', 's5', 's6']
Target — min=25  max=346  mean=152.1


In [ ]:



indices = np.random.permutation(N)


n_train = int(0.8 * N)
n_test  = N - n_train

train_idx = indices[:n_train]
test_idx  = indices[n_train:]

X_train, y_train = X_all[train_idx], y_all[train_idx]
X_test,  y_test  = X_all[test_idx],  y_all[test_idx]

print(f"Training set : {X_train.shape}")
print(f"Test set     : {X_test.shape}")

Training set : (353, 10)
Test set     : (89, 10)


In [ ]:
class DecisionTree:
    def __init__(self):

        self.tree_ = None


    def _variance_reduction(self, y_parent, y_left, y_right):

        n   = len(y_parent)
        n_L = len(y_left)
        n_R = len(y_right)

        var_parent = np.var(y_parent)

        var_L = np.var(y_left)  if n_L > 0 else 0.0
        var_R = np.var(y_right) if n_R > 0 else 0.0

        weighted_child_var = (n_L / n) * var_L + (n_R / n) * var_R
        return var_parent - weighted_child_var


    def _best_split(self, X, y, feature_indices):

        best_score = -np.inf
        best_feat  = None
        best_thr   = None

        for feat in feature_indices:
            col         = X[:, feat]
            unique_vals = np.unique(col)

            for i in range(len(unique_vals) - 1):
                thr = (unique_vals[i] + unique_vals[i + 1]) / 2.0


                left_mask  = col <= thr
                right_mask = ~left_mask
                y_left  = y[left_mask]
                y_right = y[right_mask]


                if len(y_left) == 0 or len(y_right) == 0:
                    continue

                score = self._variance_reduction(y, y_left, y_right)

                if score > best_score:
                    best_score = score
                    best_feat  = feat
                    best_thr   = thr

        return best_feat, best_thr


    def _build(self, X, y, depth, max_depth, min_samples_split, feature_indices):

        n = len(y)

        if (len(np.unique(y)) == 1
                or n < min_samples_split
                or (max_depth is not None and depth >= max_depth)):

            return {'leaf': True, 'value': float(np.mean(y))}


        feat, thr = self._best_split(X, y, feature_indices)


        if feat is None:
            return {'leaf': True, 'value': float(np.mean(y))}


        left_mask  = X[:, feat] <= thr
        right_mask = ~left_mask
        left_child  = self._build(X[left_mask],  y[left_mask],
                                  depth + 1, max_depth, min_samples_split, feature_indices)
        right_child = self._build(X[right_mask], y[right_mask],
                                  depth + 1, max_depth, min_samples_split, feature_indices)


        return {
            'leaf'      : False,
            'feature'   : feat,
            'threshold' : thr,
            'left'      : left_child,
            'right'     : right_child
        }

    def _predict_one(self, node, x):

        if node['leaf']:
            return node['value']

        if x[node['feature']] <= node['threshold']:
            return self._predict_one(node['left'],  x)
        else:
            return self._predict_one(node['right'], x)


    def build_tree(
        self,
        x: list,
        y: list,
        max_depth: int = None,
        min_samples_split: int = 2,
        feature_indices=None
    ):
        X = np.array(x, dtype=float)
        Y = np.array(y, dtype=float)

        if feature_indices is None:
            feature_indices = list(range(X.shape[1]))

        self.tree_ = self._build(X, Y,
                                 depth=0,
                                 max_depth=max_depth,
                                 min_samples_split=min_samples_split,
                                 feature_indices=feature_indices)


    def predict(self, x: list) -> float:

        x = np.array(x, dtype=float)
        return self._predict_one(self.tree_, x)

    def predict_batch(self, X):
        return np.array([self.predict(row) for row in X])


print("DecisionTree class defined.")

DecisionTree class defined.


In [ ]:
class RandomForest:


    def __init__(
        self,
        n_trees:           int   = 100,
        max_depth:         int   = None,
        min_samples_split: int   = 2,
        n_features:        int   = None,
        sample_fraction:   float = 1.0
    ):
        self.n_trees           = n_trees
        self.max_depth         = max_depth
        self.min_samples_split = min_samples_split
        self.n_features        = n_features
        self.sample_fraction   = sample_fraction
    def build_forest(self, x: list, y: list):
        X = np.array(x, dtype=float)
        Y = np.array(y, dtype=float)
        n_samples, n_total_features = X.shape

        if self.n_features is None:
            k_features = max(1, int(np.floor(np.sqrt(n_total_features))))
        else:
            k_features = self.n_features


        n_subsample = max(1, int(self.sample_fraction * n_samples))

        self.trees_ = []

        for t in range(self.n_trees):


            row_idx = np.random.choice(n_samples, size=n_subsample, replace=True)

            X_sub = X[row_idx]
            Y_sub = Y[row_idx]


            feat_idx = np.random.choice(n_total_features, size=k_features, replace=False)


            tree = DecisionTree()
            tree.build_tree(
                X_sub.tolist(), Y_sub.tolist(),
                max_depth         = self.max_depth,
                min_samples_split = self.min_samples_split,
                feature_indices   = feat_idx.tolist()
            )
            self.trees_.append(tree)

        print(f"Forest of {self.n_trees} trees built  "
              f"(each uses {k_features}/{n_total_features} features, "
              f"{n_subsample}/{n_samples} samples).")

    def predict(self, x: list) -> float:

        individual_preds = [tree.predict(x) for tree in self.trees_]
        return float(np.mean(individual_preds))


    def predict_batch(self, X):

        return np.array([self.predict(row) for row in X])


print("RandomForest class defined.")

RandomForest class defined.


In [ ]:

def mse(y_true, y_pred):
    return float(np.mean((np.array(y_true) - np.array(y_pred)) ** 2))

In [ ]:


dt = DecisionTree()

dt.build_tree(
    X_train.tolist(), y_train.tolist(),
    max_depth=5,
    min_samples_split=5
)
print("Decision tree built.")

dt_train_pred = dt.predict_batch(X_train)
dt_test_pred  = dt.predict_batch(X_test)

print("\n Single Decision Tree :")
print(f"  Train  MSE  = {mse(y_train, dt_train_pred):7.2f}     "
      f"Test  MSE  = {mse(y_test, dt_test_pred):7.2f}")

Decision tree built.

 Single Decision Tree :
  Train  MSE  = 1852.91     Test  MSE  = 4897.56


In [ ]:

np.random.seed(SEED)

rf = RandomForest(
    n_trees           = 100,
    max_depth         = 5,
    min_samples_split = 5,
    n_features        = None,
    sample_fraction   = 1.0
)
rf.build_forest(X_train.tolist(), y_train.tolist())

rf_train_pred = rf.predict_batch(X_train)
rf_test_pred  = rf.predict_batch(X_test)

print("\n  Random Forest (100 trees) :")
print(f"  Train  MSE  = {mse(y_train, rf_train_pred):7.2f}     "
      f"Test  MSE  = {mse(y_test, rf_test_pred):7.2f}")


Forest of 100 trees built  (each uses 3/10 features, 353/353 samples).

  Random Forest (100 trees) :
  Train  MSE  = 2225.05     Test  MSE  = 3616.38


In [ ]:

np.random.seed(SEED)

tree_counts  = [1, 5, 10, 20, 50, 100]
mse_by_trees = []

for n_t in tree_counts:
    rf_temp = RandomForest(n_trees=n_t, max_depth=5, min_samples_split=5)
    rf_temp.build_forest(X_train.tolist(), y_train.tolist())
    preds = rf_temp.predict_batch(X_test)
    mse_by_trees.append(mse(y_test, preds))
    print(f"  n_trees={n_t:4d}  →  Test MSE = {mse_by_trees[-1]:.2f}")

print("\nTest MSE stabilises as we add more trees.")

Forest of 1 trees built  (each uses 3/10 features, 353/353 samples).
  n_trees=   1  →  Test MSE = 5887.55
Forest of 5 trees built  (each uses 3/10 features, 353/353 samples).
  n_trees=   5  →  Test MSE = 3717.33
Forest of 10 trees built  (each uses 3/10 features, 353/353 samples).
  n_trees=  10  →  Test MSE = 4098.92
Forest of 20 trees built  (each uses 3/10 features, 353/353 samples).
  n_trees=  20  →  Test MSE = 3755.18
Forest of 50 trees built  (each uses 3/10 features, 353/353 samples).
  n_trees=  50  →  Test MSE = 3656.64
Forest of 100 trees built  (each uses 3/10 features, 353/353 samples).
  n_trees= 100  →  Test MSE = 3576.29

Test MSE stabilises as we add more trees.
